In [12]:
import sys
sys.path.append("../src")

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,log_loss, brier_score_loss
from optbinning import BinningProcess

import statsmodels.api as sm

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROCESSED = Path("../data/processed")
ARTEFACTS = Path("../outputs/models")
ARTEFACTS.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 22

In [2]:
# IMPORTANT: we load preprocessed_train (raw values), NOT woe_train.
# WoE must be refitted inside each CV fold, so we need the untransformed
# feature values as the starting point.

df = pd.read_parquet(PROCESSED / "preprocessed_train.parquet")
scorecard_features = joblib.load(ARTEFACTS / "scorecard_features.pkl")

print(f"Preprocessed train: {df.shape}")
print(f"Scorecard features: {len(scorecard_features)}")

# Verify all selected features exist in the raw preprocessed data
missing = [f for f in scorecard_features if f not in df.columns]
print(f"Missing from preprocessed: {missing}")

Preprocessed train: (307511, 249)
Scorecard features: 27
Missing from preprocessed: []


In [3]:
X = df[scorecard_features]
y = df["TARGET"]
ids = df["SK_ID_CURR"]

X_dev, X_holdout, y_dev, y_holdout, ids_dev, ids_holdout = train_test_split(
    X, y, ids,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Development set: {X_dev.shape}  bad rate {y_dev.mean():.4f}")
print(f"Holdout set:     {X_holdout.shape}  bad rate {y_holdout.mean():.4f}")

# Persist the holdout indices so the split is reproducible and auditable
joblib.dump(
    {"dev_ids": ids_dev.values, "holdout_ids": ids_holdout.values},
    ARTEFACTS / "holdout_split.pkl",
)

Development set: (246008, 27)  bad rate 0.0807
Holdout set:     (61503, 27)  bad rate 0.0807


['../outputs/models/holdout_split.pkl']

In [5]:
def build_binning(features, cat_features):
    """Fresh BinningProcess with the same config used in Phase 4."""
    fit_params = {}
    for v in features:
        if v in cat_features:
            fit_params[v] = {"min_bin_size": 0.05, "max_n_bins": 6}
        else:
            fit_params[v] = {
                "monotonic_trend": "auto_asc_desc",
                "min_bin_size": 0.05,
                "max_n_bins": 6,
            }
    return BinningProcess(
        variable_names=features,
        categorical_variables=cat_features,
        binning_fit_params=fit_params,
        n_jobs=-1,
    )


# Identify categoricals among the 27 selected features
cat_features = [
    c for c in scorecard_features
    if X_dev[c].dtype == "object"
    or X_dev[c].dtype == "bool"
    or str(X_dev[c].dtype) in ("string", "category")
]
print(f"Categorical among selected: {cat_features}\n")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_dev, y_dev), start=1):
    X_tr, X_va = X_dev.iloc[tr_idx], X_dev.iloc[va_idx]
    y_tr, y_va = y_dev.iloc[tr_idx], y_dev.iloc[va_idx]

    # --- WoE fitted on TRAIN FOLD ONLY ---
    bp = build_binning(scorecard_features, cat_features)
    bp.fit(X_tr, y_tr)

    X_tr_woe = bp.transform(X_tr)
    X_va_woe = bp.transform(X_va)

    # --- Logistic regression on WoE ---
    lr = LogisticRegression(
        C=np.inf,
        max_iter=1000,
        random_state=RANDOM_STATE,
    )
    lr.fit(X_tr_woe, y_tr)

    auc_tr = roc_auc_score(y_tr, lr.predict_proba(X_tr_woe)[:, 1])
    auc_va = roc_auc_score(y_va, lr.predict_proba(X_va_woe)[:, 1])
    fold_scores.append({"fold": fold, "auc_train": auc_tr, "auc_valid": auc_va})

    print(f"Fold {fold}:  train AUC {auc_tr:.4f}   valid AUC {auc_va:.4f}")

cv_results = pd.DataFrame(fold_scores)
print(f"\nMean valid AUC: {cv_results['auc_valid'].mean():.4f} "
      f"(+/- {cv_results['auc_valid'].std():.4f})")
print(f"Mean Gini:      {2 * cv_results['auc_valid'].mean() - 1:.4f}")

Categorical among selected: ['OCCUPATION_TYPE']

Fold 1:  train AUC 0.7503   valid AUC 0.7480
Fold 2:  train AUC 0.7501   valid AUC 0.7482
Fold 3:  train AUC 0.7507   valid AUC 0.7446
Fold 4:  train AUC 0.7509   valid AUC 0.7439
Fold 5:  train AUC 0.7496   valid AUC 0.7502

Mean valid AUC: 0.7470 (+/- 0.0027)
Mean Gini:      0.4940


In [6]:
# Fit binning on the full development set for coefficient inspection
bp_dev = build_binning(scorecard_features, cat_features)
bp_dev.fit(X_dev, y_dev)
X_dev_woe = bp_dev.transform(X_dev)

# statsmodels for p-values and standard errors
X_sm = sm.add_constant(X_dev_woe)
logit = sm.Logit(y_dev, X_sm).fit(disp=0)

coef_table = pd.DataFrame({
    "feature": X_sm.columns,
    "coef": logit.params.values,
    "std_err": logit.bse.values,
    "z": logit.tvalues.values,
    "p_value": logit.pvalues.values,
})
coef_table["significant"] = coef_table["p_value"] < 0.05
coef_table["wrong_sign"] = (coef_table["coef"] < 0) & (coef_table["feature"] != "const")

print(f"Pseudo R-squared: {logit.prsquared:.4f}\n")
print(coef_table.sort_values("p_value").to_string(index=False))

print(f"\nNon-significant (p >= 0.05): {(~coef_table['significant']).sum()}")
print(f"Wrong sign (negative coef):  {coef_table['wrong_sign'].sum()}")

Pseudo R-squared: 0.1123

                        feature    coef  std_err         z  p_value  significant  wrong_sign
                          const -2.4562   0.0118 -207.5320   0.0000         True       False
                  EXT_1_2_3_MIN -0.6257   0.0158  -39.6153   0.0000         True        True
                CREDIT_TO_GOODS -0.5625   0.0270  -20.8089   0.0000         True        True
                   EXT_1_3_MEAN -0.3412   0.0166  -20.5368   0.0000         True        True
                OCCUPATION_TYPE -0.6210   0.0330  -18.8063   0.0000         True        True
                   REFUSAL_RATE -0.5265   0.0288  -18.2564   0.0000         True        True
       MEAN_CREDIT_TO_APP_RATIO -0.5402   0.0302  -17.8596   0.0000         True        True
        INST_MEAN_PAYMENT_RATIO -0.5945   0.0367  -16.2148   0.0000         True        True
           DEBT_TO_CREDIT_RATIO -0.4118   0.0275  -14.9948   0.0000         True        True
                 YEARS_EMPLOYED -0.4315   0.

In [7]:
def fit_and_check(features, X_raw, y, cats):
    bp = build_binning(features, [c for c in cats if c in features])
    bp.fit(X_raw[features], y)
    Xw = bp.transform(X_raw[features])
    m = sm.Logit(y, sm.add_constant(Xw)).fit(disp=0)
    tbl = pd.DataFrame({
        "feature": sm.add_constant(Xw).columns,
        "coef": m.params.values,
        "p_value": m.pvalues.values,
    })
    tbl["issue"] = np.where(
        (tbl["feature"] != "const") & (tbl["coef"] > 0), "WRONG_SIGN",
        np.where(tbl["p_value"] >= 0.05, "NOT_SIG", "")
    )
    return m, tbl, bp


current = [f for f in scorecard_features
           if f not in ("EXT_SOURCE_1", "NONLIVINGAREA_AVG", "EXT_2_3_STD")]

model_v2, tbl_v2, bp_v2 = fit_and_check(current, X_dev, y_dev, cat_features)

print(f"Features: {len(current)}")
print(f"Pseudo R-squared: {model_v2.prsquared:.4f}")
print(f"Dev AUC: {roc_auc_score(y_dev, model_v2.predict(sm.add_constant(bp_v2.transform(X_dev[current])))):.4f}\n")
print(tbl_v2[tbl_v2['issue'] != ''].to_string(index=False))
print(f"\nRemaining issues: {(tbl_v2['issue'] != '').sum()}")

Features: 24
Pseudo R-squared: 0.1118
Dev AUC: 0.7498

Empty DataFrame
Columns: [feature, coef, p_value, issue]
Index: []

Remaining issues: 0


In [8]:
final_features = current

cv_scores = []
for fold, (tr_idx, va_idx) in enumerate(cv.split(X_dev, y_dev), start=1):
    X_tr, X_va = X_dev.iloc[tr_idx], X_dev.iloc[va_idx]
    y_tr, y_va = y_dev.iloc[tr_idx], y_dev.iloc[va_idx]

    bp = build_binning(final_features, [c for c in cat_features if c in final_features])
    bp.fit(X_tr[final_features], y_tr)

    X_tr_woe = bp.transform(X_tr[final_features])
    X_va_woe = bp.transform(X_va[final_features])

    lr = LogisticRegression(C=np.inf, max_iter=1000, random_state=RANDOM_STATE)
    lr.fit(X_tr_woe, y_tr)

    auc_va = roc_auc_score(y_va, lr.predict_proba(X_va_woe)[:, 1])
    cv_scores.append(auc_va)
    print(f"Fold {fold}: valid AUC {auc_va:.4f}")

print(f"\n27 features -> mean valid AUC 0.7470")
print(f"24 features -> mean valid AUC {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")
print(f"Gini: {2 * np.mean(cv_scores) - 1:.4f}")

Fold 1: valid AUC 0.7473
Fold 2: valid AUC 0.7480
Fold 3: valid AUC 0.7445
Fold 4: valid AUC 0.7438
Fold 5: valid AUC 0.7500

27 features -> mean valid AUC 0.7470
24 features -> mean valid AUC 0.7467 (+/- 0.0023)
Gini: 0.4935


In [9]:
# --- Final scorecard model on full development set ---
bp_final = build_binning(final_features, [c for c in cat_features if c in final_features])
bp_final.fit(X_dev[final_features], y_dev)

X_dev_woe_final = bp_final.transform(X_dev[final_features])

lr_final = LogisticRegression(C=np.inf, max_iter=1000, random_state=RANDOM_STATE)
lr_final.fit(X_dev_woe_final, y_dev)

# statsmodels version retained for the coefficient table used in Phase 8
logit_final = sm.Logit(y_dev, sm.add_constant(X_dev_woe_final)).fit(disp=0)

print(f"Development AUC: {roc_auc_score(y_dev, lr_final.predict_proba(X_dev_woe_final)[:, 1]):.4f}")
print(f"Pseudo R-squared: {logit_final.prsquared:.4f}")
print(f"Features: {len(final_features)}")

# --- Persist artefacts ---
joblib.dump(bp_final,        ARTEFACTS / "scorecard_binning.pkl")
joblib.dump(lr_final,        ARTEFACTS / "scorecard_model.pkl")
joblib.dump(final_features,  ARTEFACTS / "scorecard_features_final.pkl")
joblib.dump(logit_final,     ARTEFACTS / "scorecard_logit_sm.pkl")

print("\nSaved: scorecard_binning.pkl, scorecard_model.pkl,")
print("       scorecard_features_final.pkl, scorecard_logit_sm.pkl")

Development AUC: 0.7497
Pseudo R-squared: 0.1118
Features: 24

Saved: scorecard_binning.pkl, scorecard_model.pkl,
       scorecard_features_final.pkl, scorecard_logit_sm.pkl


In [11]:
# --- Load challenger data, apply the SAME holdout split ---
challenger = pd.read_parquet(PROCESSED / "challenger_train.parquet")
split = joblib.load(ARTEFACTS / "holdout_split.pkl")

ch_dev     = challenger[challenger["SK_ID_CURR"].isin(split["dev_ids"])].copy()
ch_holdout = challenger[challenger["SK_ID_CURR"].isin(split["holdout_ids"])].copy()

ch_feats = [c for c in challenger.columns if c not in ("SK_ID_CURR", "TARGET")]

Xc_dev, yc_dev = ch_dev[ch_feats], ch_dev["TARGET"]
Xc_hold, yc_hold = ch_holdout[ch_feats], ch_holdout["TARGET"]

# LightGBM needs explicit category dtype
cat_ch = Xc_dev.select_dtypes(include=["object", "bool", "category"]).columns.tolist()
for c in cat_ch:
    Xc_dev[c]  = Xc_dev[c].astype("category")
    Xc_hold[c] = Xc_hold[c].astype("category").cat.set_categories(Xc_dev[c].cat.categories)

print(f"Challenger dev:     {Xc_dev.shape}   bad rate {yc_dev.mean():.4f}")
print(f"Challenger holdout: {Xc_hold.shape}   bad rate {yc_hold.mean():.4f}")
print(f"Categorical features: {len(cat_ch)}")

# --- Calibration comparison: weighted vs unweighted ---
def eval_config(name, params):
    aucs, lls, briers = [], [], []
    for tr_idx, va_idx in cv.split(Xc_dev, yc_dev):
        X_tr, X_va = Xc_dev.iloc[tr_idx], Xc_dev.iloc[va_idx]
        y_tr, y_va = yc_dev.iloc[tr_idx], yc_dev.iloc[va_idx]

        m = lgb.LGBMClassifier(**params)
        m.fit(X_tr, y_tr,
              eval_set=[(X_va, y_va)],
              eval_metric="auc",
              callbacks=[lgb.early_stopping(100, first_metric_only=True, verbose=False)])

        p = m.predict_proba(X_va)[:, 1]
        aucs.append(roc_auc_score(y_va, p))
        lls.append(log_loss(y_va, p))
        briers.append(brier_score_loss(y_va, p))

    print(f"{name:<22} AUC {np.mean(aucs):.4f}   "
          f"LogLoss {np.mean(lls):.4f}   Brier {np.mean(briers):.5f}")
    return np.mean(aucs)


base = dict(
    n_estimators=2000, learning_rate=0.05, num_leaves=31,
    min_child_samples=100, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
    metric="auc", first_metric_only=True,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)

eval_config("unweighted",  {**base})
eval_config("is_unbalance", {**base, "is_unbalance": True})

/var/folders/wm/w2p96dd96m55kbl8s4g3_dv80000gn/T/ipykernel_7911/214978358.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Xc_dev[c]  = Xc_dev[c].astype("category")
/var/folders/wm/w2p96dd96m55kbl8s4g3_dv80000gn/T/ipykernel_7911/214978358.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Xc_hold[c] = Xc_hold[c].astype("category").cat.set_categories(Xc_dev[c].cat.categories)
/var/folders/wm/w2p96dd96m55kbl8s4g3_dv80000gn/T/ipykernel_7911/214978358.py:16: SettingWithCopyWarning: 
A value is trying to b

Challenger dev:     (246008, 247)   bad rate 0.0807
Challenger holdout: (61503, 247)   bad rate 0.0807
Categorical features: 14
unweighted             AUC 0.7852   LogLoss 0.2377   Brier 0.06604
is_unbalance           AUC 0.7841   LogLoss 0.4976   Brier 0.16560


np.float64(0.7841381224456598)

In [13]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = dict(
        n_estimators=2000,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 16, 128),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 20, 300),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        metric="auc", first_metric_only=True,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    )

    aucs = []
    for tr_idx, va_idx in cv.split(Xc_dev, yc_dev):
        X_tr, X_va = Xc_dev.iloc[tr_idx], Xc_dev.iloc[va_idx]
        y_tr, y_va = yc_dev.iloc[tr_idx], yc_dev.iloc[va_idx]

        m = lgb.LGBMClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="auc",
              callbacks=[lgb.early_stopping(100, first_metric_only=True, verbose=False)])
        aucs.append(roc_auc_score(y_va, m.predict_proba(X_va)[:, 1]))

    return np.mean(aucs)


study = optuna.create_study(direction="maximize", study_name="lgbm_challenger")
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f"\nBest CV AUC: {study.best_value:.4f}")
print(f"Baseline:    0.7852")
print("\nBest params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/20 [00:00<?, ?it/s]


Best CV AUC: 0.7881
Baseline:    0.7852

Best params:
  learning_rate: 0.025416780634235882
  num_leaves: 49
  max_depth: 4
  min_child_samples: 254
  subsample: 0.7481559363125
  colsample_bytree: 0.46705891589514903
  reg_alpha: 0.20685896899267522
  reg_lambda: 1.306690292230389


In [14]:
best_params = dict(
    **study.best_params,
    n_estimators=2000,
    subsample_freq=1,
    metric="auc",
    first_metric_only=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

# Inner split for early stopping (holdout stays untouched)
X_fit, X_es, y_fit, y_es = train_test_split(
    Xc_dev, yc_dev, test_size=0.15, stratify=yc_dev, random_state=RANDOM_STATE
)

lgb_final = lgb.LGBMClassifier(**best_params)
lgb_final.fit(X_fit, y_fit,
              eval_set=[(X_es, y_es)], eval_metric="auc",
              callbacks=[lgb.early_stopping(100, first_metric_only=True, verbose=False)])

print(f"Best iteration: {lgb_final.best_iteration_}")

# --- HOLDOUT EVALUATION — both tracks ---
p_ch = lgb_final.predict_proba(Xc_hold)[:, 1]

X_hold_woe = bp_final.transform(X_holdout[final_features])
p_sc = lr_final.predict_proba(X_hold_woe)[:, 1]

for name, p in [("Scorecard (24 feat)", p_sc), ("Challenger (247 feat)", p_ch)]:
    auc = roc_auc_score(y_holdout, p)
    print(f"{name:<24} AUC {auc:.4f}   Gini {2*auc-1:.4f}   "
          f"Brier {brier_score_loss(y_holdout, p):.5f}   "
          f"LogLoss {log_loss(y_holdout, p):.4f}")

joblib.dump(lgb_final, ARTEFACTS / "challenger_model.pkl")
joblib.dump(study.best_params, ARTEFACTS / "challenger_params.pkl")

Best iteration: 1452
Scorecard (24 feat)      AUC 0.7424   Gini 0.4848   Brier 0.06905   LogLoss 0.2512
Challenger (247 feat)    AUC 0.4987   Gini -0.0026   Brier 0.08276   LogLoss 0.3276


['../outputs/models/challenger_params.pkl']

In [15]:
print(f"y_holdout rows: {len(y_holdout)}   yc_hold rows: {len(yc_hold)}")
print(f"Bad rate y_holdout: {y_holdout.mean():.4f}   yc_hold: {yc_hold.mean():.4f}")

# Are the ID sequences identical, in the same order?
print(f"Same IDs, same order: {(ids_holdout.values == ch_holdout['SK_ID_CURR'].values).all()}")

# Score the challenger against ITS OWN aligned target
print(f"\nChallenger vs yc_hold: AUC {roc_auc_score(yc_hold, p_ch):.4f}")

y_holdout rows: 61503   yc_hold rows: 61503
Bad rate y_holdout: 0.0807   yc_hold: 0.0807
Same IDs, same order: False

Challenger vs yc_hold: AUC 0.7847


In [16]:
sc_pred = pd.DataFrame({
    "SK_ID_CURR": ids_holdout.values,
    "p_scorecard": p_sc,
    "TARGET": y_holdout.values,
})
ch_pred = pd.DataFrame({
    "SK_ID_CURR": ch_holdout["SK_ID_CURR"].values,
    "p_challenger": p_ch,
})

comparison = sc_pred.merge(ch_pred, on="SK_ID_CURR", how="inner")
print(f"Merged rows: {len(comparison)} (expected 61503)\n")

y_h = comparison["TARGET"]
for name, col in [("Scorecard (24 feat)", "p_scorecard"),
                  ("Challenger (247 feat)", "p_challenger")]:
    p = comparison[col]
    auc = roc_auc_score(y_h, p)
    print(f"{name:<24} AUC {auc:.4f}   Gini {2*auc-1:.4f}   "
          f"Brier {brier_score_loss(y_h, p):.5f}   LogLoss {log_loss(y_h, p):.4f}")

comparison.to_parquet(PROCESSED / "holdout_predictions.parquet", index=False)
print("\nSaved: holdout_predictions.parquet")

Merged rows: 61503 (expected 61503)

Scorecard (24 feat)      AUC 0.7424   Gini 0.4848   Brier 0.06905   LogLoss 0.2512
Challenger (247 feat)    AUC 0.7847   Gini 0.5694   Brier 0.06626   LogLoss 0.2382

Saved: holdout_predictions.parquet
